# Phase 2 — SQL Analysis
## E-Commerce Revenue & Customer Lifecycle Analytics
**Analyst:** Utsav Khadka  
**Dataset:** UCI Online Retail II (Year 2010-2011)  
**Engine:** DuckDB (runs SQL directly on CSV files)

---

### How This Notebook Works
Each section runs a standalone `.sql` file using DuckDB.  
The SQL files are in the `/sql/` folder — open them side by side to read the queries.

**SQL Files:**
- `01_data_exploration.sql` — Dataset profiling and overview
- `02_revenue_analysis.sql` — Monthly revenue trends and MoM growth
- `03_customer_cohorts.sql` — Cohort retention analysis
- `04_rfm_segmentation.sql` — RFM customer segmentation
- `05_product_geo_analysis.sql` — Product and geographic deep dives

---
## Setup — Import DuckDB and Configure Display

In [1]:
# WHY: DuckDB lets us run SQL directly on CSV files
# No database server needed — reads the file directly
import duckdb
import pandas as pd

# Display all columns when printing DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_rows', 50)

# Path to our cleaned dataset
CLEANED_DATA = '../data/cleaned/online_retail_cleaned.csv'

print('DuckDB version:', duckdb.__version__)
print('Setup complete. Ready to run SQL.')

DuckDB version: 1.5.2
Setup complete. Ready to run SQL.


---
## Helper Function — Run SQL Queries

**WHY:** Instead of repeating `duckdb.query().df()` every time,  
we create one function that runs any SQL and returns a clean DataFrame.

In [2]:
def run_sql(query, title=None):
    """
    Run a SQL query using DuckDB and return results as a DataFrame.
    
    Parameters:
        query (str): SQL query to execute
        title (str): Optional title to print before results
    
    Returns:
        pd.DataFrame: Query results
    """
    if title:
        print(f'\n=== {title} ===')
        print('=' * (len(title) + 8))
    
    result = duckdb.query(query).df()
    return result

print('Helper function ready.')

Helper function ready.


---
# FILE 1: Data Exploration
**SQL File:** `sql/01_data_exploration.sql`  
**Purpose:** Understand what the data contains before deeper analysis  
**Business Question:** What is the overall scale of this business?

### Query 1 — Dataset Overview

**WHY:** Before answering any business question, confirm what the data contains.  
Row counts, date range, and unique counts are the first thing any analyst checks.

**New SQL Concept:** `COUNT(DISTINCT column)` counts unique values, not total rows.  
Example: If customer 17850 placed 10 orders, `COUNT(CustomerID)` = 10 but `COUNT(DISTINCT CustomerID)` = 1

In [3]:
query1 = f"""
SELECT
    COUNT(*)                                    AS total_rows,
    COUNT(DISTINCT InvoiceNo)                   AS unique_invoices,
    COUNT(DISTINCT CustomerID)                  AS unique_customers,
    COUNT(DISTINCT StockCode)                   AS unique_products,
    COUNT(DISTINCT Country)                     AS unique_countries,
    MIN(InvoiceDate)                            AS earliest_date,
    MAX(InvoiceDate)                            AS latest_date
FROM read_csv_auto('{CLEANED_DATA}')
"""

df1 = run_sql(query1, 'Dataset Overview')
display(df1)


=== Dataset Overview ===


,total_rows,unique_invoices,unique_customers,unique_products,unique_countries,earliest_date,latest_date
0,534130,23796,4371,3938,38,2010-12-01 08:26:00,2011-12-09 12:50:00


### Query 2 — Revenue Overview

**WHY:** Total revenue, AOV (Average Order Value), and order range are the 3 numbers  
the VP will ask first. AOV = Total Revenue ÷ Total Orders.

**New SQL Concept:** CTEs (`WITH` statement)  
We first calculate revenue per order (one invoice = one order), then summarize.  
This is cleaner than nesting subqueries inside subqueries.

In [4]:
query2 = f"""
WITH order_revenue AS (
    -- Step 1: Sum revenue per invoice (one order = multiple product rows)
    SELECT
        InvoiceNo,
        CustomerID,
        SUM(Revenue) AS order_total
    FROM read_csv_auto('{CLEANED_DATA}')
    WHERE is_cancelled = false
      AND CustomerID IS NOT NULL
    GROUP BY InvoiceNo, CustomerID
)
-- Step 2: Calculate business-level metrics across all orders
SELECT
    ROUND(SUM(order_total), 2)          AS total_revenue,
    COUNT(DISTINCT InvoiceNo)           AS total_orders,
    ROUND(AVG(order_total), 2)          AS avg_order_value,
    ROUND(MIN(order_total), 2)          AS min_order_value,
    ROUND(MAX(order_total), 2)          AS max_order_value
FROM order_revenue
"""

df2 = run_sql(query2, 'Revenue Overview')
display(df2)


=== Revenue Overview ===


,total_revenue,total_orders,avg_order_value,min_order_value,max_order_value
0,"8,887,226.89",18532,479.56,0.38,"168,469.60"


### Query 3 — Top 10 Countries by Revenue

**WHY:** Answers the VP's question: *"Which regions are driving growth?"*  
We also calculate each country's % share of total revenue.

**New SQL Concept:** Window Function — `SUM() OVER ()`  
- Regular `SUM(Revenue)` = sum per group (per country)  
- `SUM(SUM(Revenue)) OVER ()` = grand total across ALL countries  
- Dividing the two gives you the percentage share — without a second query

In [5]:
query3 = f"""
SELECT
    Country,
    COUNT(DISTINCT InvoiceNo)           AS total_orders,
    COUNT(DISTINCT CustomerID)          AS total_customers,
    ROUND(SUM(Revenue), 2)              AS total_revenue,
    -- Window function: calculates % share of grand total revenue
    ROUND(
        SUM(Revenue) * 100.0 /
        SUM(SUM(Revenue)) OVER (),
        2
    )                                   AS revenue_pct
FROM read_csv_auto('{CLEANED_DATA}')
WHERE is_cancelled = false
GROUP BY Country
ORDER BY total_revenue DESC
LIMIT 10
"""

df3 = run_sql(query3, 'Top 10 Countries by Revenue')
display(df3)


=== Top 10 Countries by Revenue ===


,Country,total_orders,total_customers,total_revenue,revenue_pct
0,United Kingdom,18019,3920,"9,001,744.09",84.59
1,Netherlands,94,9,"285,446.34",2.68
2,EIRE,288,3,"283,140.52",2.66
3,Germany,457,94,"228,678.40",2.15
4,France,392,87,"209,643.37",1.97
5,Australia,57,9,"138,453.81",1.30
6,Spain,90,30,"61,558.56",0.58
7,Switzerland,54,21,"57,067.60",0.54
8,Belgium,98,25,"41,196.34",0.39
9,Sweden,36,8,"38,367.83",0.36


### Query 4 — Top 10 Products by Revenue

**WHY:** Answers: *"Which products are driving growth?"*  
Identifies your most valuable SKUs — critical for inventory and marketing decisions.

**New SQL Concept:** `RANK() OVER (ORDER BY ...)`  
Assigns a rank number to each row based on a column.  
Rank 1 = highest revenue product. No need to manually number rows.

In [6]:
query4 = f"""
SELECT
    StockCode,
    Description,
    SUM(Quantity)                           AS total_units_sold,
    ROUND(SUM(Revenue), 2)                  AS total_revenue,
    ROUND(AVG(UnitPrice), 2)                AS avg_unit_price,
    -- RANK assigns position based on revenue (1 = highest)
    RANK() OVER (ORDER BY SUM(Revenue) DESC) AS revenue_rank
FROM read_csv_auto('{CLEANED_DATA}')
WHERE is_cancelled = false
  AND Description IS NOT NULL
GROUP BY StockCode, Description
ORDER BY total_revenue DESC
LIMIT 10
"""

df4 = run_sql(query4, 'Top 10 Products by Revenue')
display(df4)


=== Top 10 Products by Revenue ===


,StockCode,Description,total_units_sold,total_revenue,avg_unit_price,revenue_rank
0,DOT,Dotcom Postage,706.00,"206,248.77",292.14,1
1,22423,Regency Cakestand 3 Tier,"13,851.00","174,156.54",13.98,2
2,23843,"Paper Craft , Little Birdie","80,995.00","168,469.60",2.08,3
3,85123A,White Hanging Heart T-Light Holder,"37,580.00","104,284.24",3.12,4
4,47566,Party Bunting,"18,283.00","99,445.23",5.80,5
5,85099B,Jumbo Bag Red Retrospot,"48,371.00","94,159.81",2.49,6
6,23166,Medium Ceramic Top Storage Jar,"78,033.00","81,700.92",1.47,7
7,POST,Postage,"3,151.00","78,119.88",31.06,8
8,M,Manual,"6,984.00","77,750.27",234.49,9
9,23084,Rabbit Night Light,"30,739.00","66,870.03",2.39,10


### Query 5 — Monthly Transaction Volume

**WHY:** Shows business scale month by month — orders, customers, revenue.  
Reveals seasonality patterns before we do the deep revenue trend analysis.

**Note:** `YearMonth` column was created in Phase 1 (format: 2010-12).  
This makes monthly grouping simple — one column, clean output.

In [7]:
query5 = f"""
SELECT
    YearMonth,
    COUNT(DISTINCT InvoiceNo)           AS total_orders,
    COUNT(DISTINCT CustomerID)          AS active_customers,
    ROUND(SUM(Revenue), 2)              AS monthly_revenue,
    ROUND(AVG(Revenue), 2)              AS avg_transaction_value
FROM read_csv_auto('{CLEANED_DATA}')
WHERE is_cancelled = false
  AND CustomerID IS NOT NULL
GROUP BY YearMonth
ORDER BY YearMonth
"""

df5 = run_sql(query5, 'Monthly Transaction Volume')
display(df5)


=== Monthly Transaction Volume ===


,YearMonth,total_orders,active_customers,monthly_revenue,avg_transaction_value
0,2010-12,1400,885,"570,422.73",22.22
1,2011-01,987,741,"568,101.31",27.07
2,2011-02,997,758,"446,084.92",22.64
3,2011-03,1321,974,"594,081.76",22.11
4,2011-04,1149,856,"468,374.33",20.88
5,2011-05,1555,1056,"677,355.15",24.13
6,2011-06,1393,991,"660,046.05",24.51
7,2011-07,1331,949,"598,962.90",22.53
8,2011-08,1280,935,"644,051.04",24.04
9,2011-09,1755,1266,"950,690.20",23.97


### Query 6 — Customer Overview

**WHY:** Understanding the customer base at a high level —  
how many customers, how often they buy, how much they spend on average.

**CTE Pattern Review:**  
Step 1 (customer_summary CTE) → calculates metrics per customer  
Step 2 (final SELECT) → summarizes those metrics across all customers  
This is cleaner than one deeply nested query.

In [8]:
query6 = f"""
WITH customer_summary AS (
    -- Step 1: Calculate per-customer metrics
    SELECT
        CustomerID,
        COUNT(DISTINCT InvoiceNo)       AS total_orders,
        ROUND(SUM(Revenue), 2)          AS total_spent,
        MIN(InvoiceDate)                AS first_purchase,
        MAX(InvoiceDate)                AS last_purchase
    FROM read_csv_auto('{CLEANED_DATA}')
    WHERE is_cancelled = false
      AND CustomerID IS NOT NULL
    GROUP BY CustomerID
)
-- Step 2: Summarize across all customers
SELECT
    COUNT(CustomerID)                   AS total_customers,
    ROUND(AVG(total_orders), 1)         AS avg_orders_per_customer,
    ROUND(AVG(total_spent), 2)          AS avg_revenue_per_customer,
    ROUND(MIN(total_spent), 2)          AS min_customer_revenue,
    ROUND(MAX(total_spent), 2)          AS max_customer_revenue
FROM customer_summary
"""

df6 = run_sql(query6, 'Customer Overview')
display(df6)


=== Customer Overview ===


,total_customers,avg_orders_per_customer,avg_revenue_per_customer,min_customer_revenue,max_customer_revenue
0,4338,4.30,"2,048.69",3.75,"280,206.02"


---
## Summary — What File 1 Told Us

After running all 6 queries, you should be able to answer:
- How many customers, products, countries does this business serve?
- What is the total revenue and average order value?
- Which country generates the most revenue?
- Which product generates the most revenue?
- Which month had the highest transaction volume?

**Next:** `02_revenue_analysis.sql` — Month-over-Month growth, running totals, moving averages

---
# FILE 2: Revenue Analysis
**SQL File:** `sql/02_revenue_analysis.sql`  
**Purpose:** Monthly revenue trends, MoM growth, running totals, moving averages  
**Business Question:** Is revenue actually growing or declining?

### New SQL Concepts in This File
| Concept | What It Does |
|---------|-------------|
| `LAG(col, 1)` | Gets the value from 1 row above (previous month) |
| `SUM() OVER (ORDER BY...)` | Running total — adds up as rows go down |
| `AVG() OVER (ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` | 3-month moving average |

### Query 1 — Monthly Revenue with MoM Growth

**WHY:** The VP said revenue looks flat. This query proves or disproves it.  
We calculate exact growth/decline percentage for every single month.

**LAG() explained:**
```
November:  £1,156,205   ← current month
LAG(1):    £1,035,642   ← looks back 1 row = October
Growth:    £120,563     ← November minus October
Growth %:  11.6%        ← (£120,563 / £1,035,642) × 100
```
First row will always have NULL for prev_month — there's no row before December 2010.

In [9]:
query_rev1 = f"""
WITH monthly_revenue AS (
    -- Step 1: Calculate total revenue per month
    SELECT
        YearMonth,
        COUNT(DISTINCT InvoiceNo)               AS total_orders,
        COUNT(DISTINCT CustomerID)              AS active_customers,
        ROUND(SUM(Revenue), 2)                  AS monthly_revenue
    FROM read_csv_auto('{CLEANED_DATA}')
    WHERE is_cancelled = false
      AND CustomerID IS NOT NULL
    GROUP BY YearMonth
),
mom_growth AS (
    -- Step 2: Use LAG() to compare current vs previous month
    SELECT
        YearMonth,
        total_orders,
        active_customers,
        monthly_revenue,
        LAG(monthly_revenue, 1) OVER (ORDER BY YearMonth) AS prev_month_revenue,
        ROUND(
            (monthly_revenue - LAG(monthly_revenue, 1) OVER (ORDER BY YearMonth))
            * 100.0
            / LAG(monthly_revenue, 1) OVER (ORDER BY YearMonth),
        2) AS mom_growth_pct
    FROM monthly_revenue
)
-- Step 3: Add running total using SUM() OVER()
SELECT
    YearMonth,
    total_orders,
    active_customers,
    monthly_revenue,
    prev_month_revenue,
    mom_growth_pct,
    ROUND(SUM(monthly_revenue) OVER (ORDER BY YearMonth), 2) AS running_total_revenue
FROM mom_growth
ORDER BY YearMonth
"""

df_rev1 = run_sql(query_rev1, 'Monthly Revenue with MoM Growth')
display(df_rev1)


=== Monthly Revenue with MoM Growth ===


,YearMonth,total_orders,active_customers,monthly_revenue,prev_month_revenue,mom_growth_pct,running_total_revenue
0,2010-12,1400,885,"570,422.73",NaN,NaN,"570,422.73"
1,2011-01,987,741,"568,101.31","570,422.73",-0.41,"1,138,524.04"
2,2011-02,997,758,"446,084.92","568,101.31",-21.48,"1,584,608.96"
3,2011-03,1321,974,"594,081.76","446,084.92",33.18,"2,178,690.72"
4,2011-04,1149,856,"468,374.33","594,081.76",-21.16,"2,647,065.05"
5,2011-05,1555,1056,"677,355.15","468,374.33",44.62,"3,324,420.20"
6,2011-06,1393,991,"660,046.05","677,355.15",-2.56,"3,984,466.25"
7,2011-07,1331,949,"598,962.90","660,046.05",-9.25,"4,583,429.15"
8,2011-08,1280,935,"644,051.04","598,962.90",7.53,"5,227,480.19"
9,2011-09,1755,1266,"950,690.20","644,051.04",47.61,"6,178,170.39"


### Query 2 — 3-Month Moving Average

**WHY:** Raw monthly revenue jumps around. Moving average reveals the real underlying trend.

**ROWS BETWEEN 2 PRECEDING AND CURRENT ROW explained:**
```
For November row:
  CURRENT ROW  = November   £1,156,205
  1 PRECEDING  = October    £1,035,642
  2 PRECEDING  = September  £950,690
  Moving Avg   = average of all 3 = £1,047,512
```
`variance_from_trend` = positive means above trend, negative means below trend.

In [10]:
query_rev2 = f"""
WITH monthly_revenue AS (
    SELECT
        YearMonth,
        ROUND(SUM(Revenue), 2) AS monthly_revenue
    FROM read_csv_auto('{CLEANED_DATA}')
    WHERE is_cancelled = false
      AND CustomerID IS NOT NULL
    GROUP BY YearMonth
)
SELECT
    YearMonth,
    monthly_revenue,
    ROUND(AVG(monthly_revenue) OVER (
        ORDER BY YearMonth
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS moving_avg_3month,
    ROUND(monthly_revenue - AVG(monthly_revenue) OVER (
        ORDER BY YearMonth
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS variance_from_trend
FROM monthly_revenue
ORDER BY YearMonth
"""

df_rev2 = run_sql(query_rev2, '3-Month Moving Average')
display(df_rev2)


=== 3-Month Moving Average ===


,YearMonth,monthly_revenue,moving_avg_3month,variance_from_trend
0,2010-12,"570,422.73","570,422.73",0.00
1,2011-01,"568,101.31","569,262.02","-1,160.71"
2,2011-02,"446,084.92","528,202.99","-82,118.07"
3,2011-03,"594,081.76","536,089.33","57,992.43"
4,2011-04,"468,374.33","502,847.00","-34,472.67"
5,2011-05,"677,355.15","579,937.08","97,418.07"
6,2011-06,"660,046.05","601,925.18","58,120.87"
7,2011-07,"598,962.90","645,454.70","-46,491.80"
8,2011-08,"644,051.04","634,353.33","9,697.71"
9,2011-09,"950,690.20","731,234.71","219,455.49"


### Query 3 — AOV Trend Over Time

**WHY:** If revenue grows but AOV declines — we need MORE orders just to stay flat.  
That's a warning sign. Are best customers spending less per visit?

In [11]:
query_rev3 = f"""
WITH order_totals AS (
    SELECT
        YearMonth,
        InvoiceNo,
        SUM(Revenue) AS order_value
    FROM read_csv_auto('{CLEANED_DATA}')
    WHERE is_cancelled = false
      AND CustomerID IS NOT NULL
    GROUP BY YearMonth, InvoiceNo
)
SELECT
    YearMonth,
    COUNT(InvoiceNo)            AS total_orders,
    ROUND(SUM(order_value), 2)  AS monthly_revenue,
    ROUND(AVG(order_value), 2)  AS avg_order_value,
    ROUND(MIN(order_value), 2)  AS min_order_value,
    ROUND(MAX(order_value), 2)  AS max_order_value
FROM order_totals
GROUP BY YearMonth
ORDER BY YearMonth
"""

df_rev3 = run_sql(query_rev3, 'AOV Trend Over Time')
display(df_rev3)


=== AOV Trend Over Time ===


,YearMonth,total_orders,monthly_revenue,avg_order_value,min_order_value,max_order_value
0,2010-12,1400,"570,422.73",407.44,0.95,"15,885.49"
1,2011-01,987,"568,101.31",575.58,0.55,"77,183.60"
2,2011-02,997,"446,084.92",447.43,2.42,"14,022.92"
3,2011-03,1321,"594,081.76",449.72,2.50,"16,726.84"
4,2011-04,1149,"468,374.33",407.64,2.10,"21,535.90"
5,2011-05,1555,"677,355.15",435.60,1.95,"14,415.74"
6,2011-06,1393,"660,046.05",473.83,1.45,"38,970.00"
7,2011-07,1331,"598,962.90",450.01,1.25,"13,464.26"
8,2011-08,1280,"644,051.04",503.16,2.25,"21,880.44"
9,2011-09,1755,"950,690.20",541.70,0.40,"31,698.16"


### Query 4 — Revenue by Day of Week

**WHY:** Operational insight — which days drive the most revenue?  
Helps plan email campaigns, promotions, and customer support staffing.

In [12]:
query_rev4 = f"""
SELECT
    Day_of_Week,
    COUNT(DISTINCT InvoiceNo)               AS total_orders,
    ROUND(SUM(Revenue), 2)                  AS total_revenue,
    ROUND(AVG(Revenue), 2)                  AS avg_revenue_per_transaction,
    RANK() OVER (ORDER BY SUM(Revenue) DESC) AS revenue_rank
FROM read_csv_auto('{CLEANED_DATA}')
WHERE is_cancelled = false
  AND CustomerID IS NOT NULL
GROUP BY Day_of_Week
ORDER BY total_revenue DESC
"""

df_rev4 = run_sql(query_rev4, 'Revenue by Day of Week')
display(df_rev4)


=== Revenue by Day of Week ===


,Day_of_Week,total_orders,total_revenue,avg_revenue_per_transaction,revenue_rank
0,Thursday,4032,"1,973,015.73",24.90,1
1,Tuesday,3184,"1,697,733.80",25.82,2
2,Wednesday,3455,"1,584,283.83",23.28,3
3,Friday,2829,"1,483,098.81",27.35,4
4,Monday,2863,"1,363,604.40",21.23,5
5,Sunday,2169,"785,490.32",12.83,6


### Query 5 — Revenue by Hour of Day

**WHY:** When are customers most active during the day?  
Peak hours = best time to send marketing emails and run promotions.

In [13]:
query_rev5 = f"""
SELECT
    Hour,
    COUNT(DISTINCT InvoiceNo)       AS total_orders,
    ROUND(SUM(Revenue), 2)          AS total_revenue,
    ROUND(AVG(Revenue), 2)          AS avg_order_value
FROM read_csv_auto('{CLEANED_DATA}')
WHERE is_cancelled = false
  AND CustomerID IS NOT NULL
GROUP BY Hour
ORDER BY Hour
"""

df_rev5 = run_sql(query_rev5, 'Revenue by Hour of Day')
display(df_rev5)


=== Revenue by Hour of Day ===


,Hour,total_orders,total_revenue,avg_order_value
0,6,1,4.25,4.25
1,7,29,"31,059.21",81.95
2,8,555,"281,997.79",32.46
3,9,1393,"842,392.34",38.42
4,10,2226,"1,259,267.59",33.34
5,11,2277,"1,101,177.60",22.77
6,12,3130,"1,373,713.39",19.36
7,13,2636,"1,168,724.20",18.55
8,14,2274,"991,992.82",18.63
9,15,2037,"963,559.68",21.51


### Query 6 — New vs Returning Customer Revenue

**WHY:** Are we growing through new customers or retaining existing ones?  
A healthy business needs both. If only new customers drive revenue — retention is broken.

**How it works:**  
- Find each customer's first purchase month  
- Tag every transaction: same month as first purchase = New, any later month = Returning

In [14]:
query_rev6 = f"""
WITH customer_first_purchase AS (
    -- Step 1: Find each customer's very first purchase month
    SELECT
        CustomerID,
        MIN(YearMonth) AS first_purchase_month
    FROM read_csv_auto('{CLEANED_DATA}')
    WHERE is_cancelled = false
      AND CustomerID IS NOT NULL
    GROUP BY CustomerID
),
tagged_orders AS (
    -- Step 2: Tag each transaction as New or Returning
    SELECT
        t.YearMonth,
        t.CustomerID,
        t.Revenue,
        CASE
            WHEN t.YearMonth = c.first_purchase_month THEN 'New Customer'
            ELSE 'Returning Customer'
        END AS customer_type
    FROM read_csv_auto('{CLEANED_DATA}') t
    LEFT JOIN customer_first_purchase c ON t.CustomerID = c.CustomerID
    WHERE t.is_cancelled = false
      AND t.CustomerID IS NOT NULL
)
-- Step 3: Summarize revenue by type per month
SELECT
    YearMonth,
    customer_type,
    COUNT(DISTINCT CustomerID)      AS customer_count,
    ROUND(SUM(Revenue), 2)          AS total_revenue
FROM tagged_orders
GROUP BY YearMonth, customer_type
ORDER BY YearMonth, customer_type
"""

df_rev6 = run_sql(query_rev6, 'New vs Returning Customer Revenue')
display(df_rev6)


=== New vs Returning Customer Revenue ===


,YearMonth,customer_type,customer_count,total_revenue
0,2010-12,New Customer,885,"570,422.73"
1,2011-01,New Customer,417,"292,366.84"
2,2011-01,Returning Customer,324,"275,734.47"
3,2011-02,New Customer,380,"157,700.59"
4,2011-02,Returning Customer,378,"288,384.33"
5,2011-03,New Customer,452,"199,619.67"
6,2011-03,Returning Customer,522,"394,462.09"
7,2011-04,New Customer,300,"121,809.05"
8,2011-04,Returning Customer,556,"346,565.28"
9,2011-05,New Customer,284,"123,739.30"


---
## Summary — What File 2 Told Us

After running all 6 queries you should be able to answer:
- Which months had positive vs negative MoM growth?
- What is the overall revenue trend (from moving average)?
- Is AOV growing or shrinking over time?
- Which day of the week drives the most revenue?
- What hours of the day are customers most active?
- Is revenue driven more by new or returning customers?

**Next:** `03_customer_cohorts.sql` — Cohort retention analysis  
This is the most complex SQL in the project — and the most impressive on a resume.